# Metacritic Videogame Scores Study
---

### Importación de las librerías

In [ ]:
import $ivy.`org.plotly-scala::plotly-almond:0.8.5`
import plotly._, element._, layout._, Almond._

import scala.io._, scala.util._
import os._

init(offline=true)

### Funciones para procesar los archivos CSV resultantes

In [ ]:
def readData(filePath: String): List[Map[String, String]] = {
    val lines = Using(Source.fromFile(filePath))(_.getLines().toList).get
    val header = lines.head.split(",").map(_.trim)
    
    lines.tail.map {
        line =>
            val values = line.split(",").map(_.trim)
            header.zip(values).toMap
    }
}

def getFilePath(subfolder: String): String = {
    val folderPath = os.pwd / "data_filtered" / "csv" / subfolder
    val files = os.list(folderPath).filter(path => path.ext == "csv" || path.last.startsWith("part"))
    
    files.headOption match {
        case Some(f) => f.toString
        case None => throw new Exception(s"No data file found in $subfolder")
    }
}

## Visualización de las Queries
---

### Evolución de puntuaciones a lo largo del tiempo

In [ ]:
val scoresMeta = readData(getFilePath("1_scores_over_time")).filter(row => Try(row("avg_metascore").toDouble).isSuccess)
val scoresUser = readData(getFilePath("1_scores_over_time")).filter(row => Try(row("avg_user_score").toDouble).isSuccess)

val yearsMeta = scoresMeta.map(_("year").toInt)
val yearsUser = scoresUser.map(_("year").toInt)
val avgMeta = scoresMeta.map(_("avg_metascore").toDouble)
val avgUser = scoresUser.map(_("avg_user_score").toDouble)

val traceMeta = Seq(Scatter().withX(yearsMeta).withY(avgMeta).withName("Metascore"))
val traceUser = Seq(Scatter().withX(yearsUser).withY(avgUser).withName("User Score"))
val layout = Layout().withXaxis(Axis().withTitle("Año")).withYaxis(Axis().withTitle("Puntuación Media"))

plot(traceMeta, layout.withTitle("Evolución de puntuaciones de la crítica a lo largo del tiempo"))
plot(traceUser, layout.withTitle("Evolución de puntuaciones de los usuarios a lo largo del tiempo"))

### Géneros mejor valorados

In [ ]:
val genresMeta = readData(getFilePath("2_best_genres_meta")).take(10)
val genresUser = readData(getFilePath("2_best_genres_user")).take(10)

val genreNamesMeta = genresMeta.map(_("genre"))
val genreNamesUser = genresUser.map(_("genre"))
val avgMeta = genresMeta.map(_("avg_metascore").toDouble)
val avgMeta2 = genresMeta.map(_("avg_user_score").toDouble * 10)
val avgUser = genresUser.map(_("avg_user_score").toDouble)
val avgUser2 = genresUser.map(_("avg_metascore").toDouble / 10)

val barMeta = Seq(Bar(genreNamesMeta, avgMeta).withName("Metascore"), Bar(genreNamesMeta, avgMeta2).withName("User Score"))
val barUser = Seq(Bar(genreNamesUser, avgUser).withName("User Score"), Bar(genreNamesUser, avgUser2).withName("Metascore"))
val layout = Layout().withXaxis(Axis().withTitle("Género")).withYaxis(Axis().withTitle("Puntuación Media")).withBarmode(BarMode.Group)

plot(traceMeta, layout.withTitle("Géneros mejor valorados por la crítica"))
plot(traceUser, layout.withTitle("Géneros mejor valorados por los usuarios"))

### Desarrolladoras con mejor media de calidad

In [ ]:
val devsMeta = readData(getFilePath("3_top_developers_meta")).take(10)
val devsUser = readData(getFilePath("3_top_developers_user")).take(10)

val devNamesMeta = devsMeta.map(_("developer"))
val devNamesUser = devsUser.map(_("developer"))
val avgMeta = devsMeta.map(_("avg_metascore").toDouble)
val avgMeta2 = devsMeta.map(_("avg_user_score").toDouble * 10)
val avgUser = devsUser.map(_("avg_user_score").toDouble)
val avgUser2 = devsUser.map(_("avg_metascore").toDouble / 10)

val barMeta = Seq(Bar(devNamesMeta, avgMeta).withName("Metascore"), Bar(devNamesMeta, avgMeta2).withName("User Score"))
val barUser = Seq(Bar(devNamesUser, avgUser).withName("User Score"), Bar(devNamesUser, avgUser2).withName("Metascore"))
val layout = Layout().withXaxis(Axis().withTitle("Desarrolladora")).withYaxis(Axis().withTitle("Puntuación Media")).withBarmode(BarMode.Group)

plot(traceMeta, layout.withTitle("Desarrolladoras con mejor calidad media según la crítica"))
plot(traceUser, layout.withTitle("Desarrolladoras con mejor calidad media según los usuarios"))

### Reseñas por año de lanzamientos

In [ ]:
val reviewsMeta = readData(getFilePath("4_reviews_per_year")).filter(row => Try(row("critic_reviews").toInt).isSuccess)
val reviewsUser = readData(getFilePath("4_reviews_per_year")).filter(row => Try(row("user_reviews").toInt).isSuccess)

val yearsMeta = reviewsMeta.map(_("year").toInt)
val yearsUser = reviewsUser.map(_("year").toInt)
val countMeta = reviewsMeta.map(_("critic_reviews").toInt)
val countUser = reviewsUser.map(_("user_reviews").toInt)

val traceMeta = Seq(Scatter().withX(yearsMeta).withY(countMeta).withName("Críticas"))
val traceUser = Seq(Scatter().withX(yearsUser).withY(countUser).withName("Usuarios"))
val layout = Layout().withXaxis(Axis().withTitle("Año")).withYaxis(Axis().withTitle("Número de Reseñas"))

plot(traceMeta, layout.withTitle("Número de reseñas de la crítica por año"))
plot(traceUser, layout.withTitle("Número de reseñas de los usuarios por año"))

### Géneros más infravalorados

In [ ]:
val underrated = readData(getFilePath("5_underrated_genres")).take(10)

val genres = underrated.map(_("genre"))
val gaps = underrated.map(_("gap").toDouble)

val bar = Seq(Bar(genres, gaps).withName("Brecha"))
val layout = Layout().withXaxis(Axis().withTitle("Género")).withYaxis(Axis().withTitle("Diferencia entre Puntuaciones"))

plot(bar, layout.withTitle("Géneros más infravalorados"))

### Polarización de usuarios por videojuego

In [ ]:
val polarizing = readData(getFilePath("6_polarizing_games")).take(10)

val games = polarizing.map(_("game"))
val polarization = polarizing.map(_("polarization").toDouble)

val bar = Seq(Bar(games, polarization).withName("Polarización"))
val layout = Layout().withXaxis(Axis().withTitle("Videojuego")).withYaxis(Axis().withTitle("Índice de Polarización").withRange(0.99, 1.0))

plot(bar, layout.withTitle("Polarización de usuarios por videojuego"))

### Discrepancia crítica vs usuario por editora

In [ ]:
val discrenpacy = readData(getFilePath("7_publisher_discrepancy")).take(10)

val publishers = discrenpacy.map(_("publisher"))
val gaps = discrenpacy.map(_("gap").toDouble)

val bar = Seq(Bar(publishers, gaps).withName("Brecha"))
val layout = Layout().withXaxis(Axis().withTitle("Editora")).withYaxis(Axis().withTitle("Diferencia Absoluta entre Puntuaciones"))

plot(bar, layout.withTitle("Discrepancia crítica vs usuario por editora"))